<a href="https://colab.research.google.com/github/israakadhem0-coder/Drug-discovery_ML/blob/main/Keap1_israa_code1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
! pip install chembl_webresource_client
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client

# 1. Target search for Keap1
target = new_client.target
target_query = target.search('Keap1')
targets = pd.DataFrame.from_dict(target_query)

# Display target list to verify selection
print("Available Targets:")
print(targets[['target_chembl_id', 'pref_name', 'organism']].head())

# 2. Select Human Keap1 (CHEMBL1075138)
selected_target = 'CHEMBL1075138'

# 3. Fetch bioactivity data for IC50
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")
df = pd.DataFrame.from_dict(res)

# 4. Clean, validate units, and handle missing values
# Keep standard_units in nM for consistent pIC50 conversion
df_clean = df[df['standard_units'] == 'nM'].copy()

# Ensure numeric values and drop missing essential fields
df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'], errors='coerce')
df_clean = df_clean.dropna(subset=['standard_value', 'canonical_smiles', 'molecule_chembl_id'])

# Remove non-positive IC50 values to avoid math errors during log conversion
df_clean = df_clean[df_clean['standard_value'] > 0]

# Deduplicate duplicate SMILES by taking the median standard_value for ML consistency
df_clean = df_clean.groupby('canonical_smiles', as_index=False).agg({
    'molecule_chembl_id': 'first',
    'standard_value': 'median'
})

# 5. Calculate pIC50 for ML regression
def norm_value(input_val):
    # Cap values at 100,000,000 nM so pIC50 values stay positive
    return 100000000 if input_val > 100000000 else input_val

def pic50(input_df):
    pIC50 = []
    for i in input_df['standard_value_norm']:
        molar = i * (10**-9)  # Convert nM to M
        pIC50.append(-np.log10(molar))
    input_df['pIC50'] = pIC50
    return input_df

df_clean['standard_value_norm'] = df_clean['standard_value'].apply(norm_value)
df_ml = pic50(df_clean)

# 6. Assign categorical labels (classification target)
conditions = [
    df_ml['standard_value'] <= 1000,
    df_ml['standard_value'] >= 10000
]
choices = ['active', 'inactive']
df_ml['bioactivity_class'] = np.select(conditions, choices, default='intermediate')

# 7. Final ML Dataframe selection & Export
df_final = df_ml[[
    'molecule_chembl_id',
    'canonical_smiles',
    'standard_value',
    'pIC50',
    'bioactivity_class'
]].reset_index(drop=True)

# Export ready-to-use dataset for ML model training
df_final.to_csv('keap1_bioactivity_data_processed_ml.csv', index=False)

print("\nProcessed Dataset Ready for ML:")
print(df_final.head())
print(f"\nTotal compounds downloaded and prepared: {len(df_final)}")

Available Targets:
  target_chembl_id                            pref_name           organism
0    CHEMBL3038498                           Keap1/Nrf2       Homo sapiens
1    CHEMBL4296095                           KEAP1/NRF2  Rattus norvegicus
2    CHEMBL4523596  Kelch-like ECH-associated protein 1  Rattus norvegicus
3    CHEMBL3562164  Kelch-like ECH-associated protein 1       Mus musculus
4    CHEMBL2069156  Kelch-like ECH-associated protein 1       Homo sapiens

Processed Dataset Ready for ML:
  molecule_chembl_id                                   canonical_smiles  \
0      CHEMBL3422392  Br.COc1cc2c(cc1O)-c1c(c3ccc([N+](=O)[O-])cc3c(...   
1      CHEMBL3422390  Br.COc1cc2c(cc1O)-c1c(c3ccc([N+](=O)[O-])cc3c(...   
2      CHEMBL3422391  Br.COc1cc2c(cc1O)-c1c(c3ccc([N+](=O)[O-])cc3c(...   
3      CHEMBL3422389  Br.COc1cc2c(cc1O)C(=O)c1c-2n(CCCN)c(=O)c2cc([N...   
4      CHEMBL3422387  Br.COc1cc2c(cc1O)C(=O)c1c-2n(CCCN2CCOCC2)c(=O)...   

   standard_value     pIC50 bioactivity_class  